In [1]:
# Useful Link: https://hep.itp.tuwien.ac.at/%7Ekreuzer/CY/CYcsws.html

#import os
#from __future__ import annotations
#from dataclasses import dataclass
#from typing import Iterable, List, Tuple, Sequence, Optional

import time
import numpy as np
import pandas as pd
#import pyarrow.dataset as ds
#from datasets import load_dataset

#import cdd
#from pypoman import compute_polytope_halfspaces, compute_polytope_vertices
#import matplotlib.pyplot as plt
from cytools import Polytope

from geometry import (
    build_55_overlap3_from_5weights, 
    canonicalize_rows, 
    lattice_points_from_matrix,
    point_in_relative_interior,
    generate_reduced_5_ws_via_x,
    analyze_cws_candidate
)

/opt/conda/lib/python3.11/site-packages/cytools/config.py:154: UserWarning: 
**************************************************************
Some of these features may be broken or not fully tested,
and they may undergo significant changes in future versions.
**************************************************************

  warnings.warn(


# Construct 5 + 5 CWS with Overlap 3

In [5]:
records_complete = generate_reduced_5_ws_via_x(
    max_depth=5, 
    verbose=False, 
    max_results=100
)

print("Anzahl gefundener reduzierter 5er-WS:", len(records_complete))

Anzahl gefundener reduzierter 5er-WS: 100


## Tabelle der reduzierten WS und der vollen Sextupel

In [6]:
df_single5_x_complete = pd.DataFrame([
    {
        "x_points": rec.x_points,
        "q_tilde": rec.q_tilde,
        "weights_5_reduced": rec.int_weights_5,
        "weights_6_full": rec.sextuple_weights,
        "degree_full": sum(rec.sextuple_weights),
    }
    for rec in records_complete
])

df_single5_x_complete.to_csv("x_based_single5_reduced_candidates.csv", index=False)
print("Gespeichert: x_based_single5_reduced_candidates.csv")
print("Anzahl Zeilen:", len(df_single5_x_complete))
df_single5_x_complete.head(20)

Gespeichert: x_based_single5_reduced_candidates.csv
Anzahl Zeilen: 100


,x_points,q_tilde,weights_5_reduced,weights_6_full,degree_full
0,"((2, 2, 2, 2, 2), (4, 0, 0, 0, 0), (3, 1, 0, 0...","(0.25, 0.25, -0.0, 0.0, 0.0)","(1, 1, 0, 0, 0)","(2, 1, 1, 0, 0, 0)",4
1,"((2, 2, 2, 2, 2), (4, 0, 0, 0, 0), (3, 2, 0, 0...","(0.25, 0.125, 0.125, -0.0, 0.0)","(2, 1, 1, 0, 0)","(4, 2, 1, 1, 0, 0)",8
2,"((2, 2, 2, 2, 2), (4, 0, 0, 0, 0), (3, 3, 0, 0...","(0.25, 0.08333333333333, 0.16666666666667, -0....","(3, 1, 2, 0, 0)","(6, 3, 1, 2, 0, 0)",12
3,"((2, 2, 2, 2, 2), (4, 0, 0, 0, 0), (3, 3, 0, 0...","(0.25, 0.08333333333333, 0.08333333333333, 0.0...","(3, 1, 1, 1, 0)","(6, 3, 1, 1, 1, 0)",12
4,"((2, 2, 2, 2, 2), (4, 0, 0, 0, 0), (3, 2, 1, 0...","(0.25, 0.125, -0.0, 0.125, -0.0)","(2, 1, 0, 1, 0)","(4, 2, 1, 0, 1, 0)",8
5,"((2, 2, 2, 2, 2), (4, 0, 0, 0, 0), (3, 2, 1, 0...","(0.25, 0.0, 0.25, -0.0, 0.0)","(1, 0, 1, 0, 0)","(2, 1, 0, 1, 0, 0)",4
6,"((2, 2, 2, 2, 2), (5, 0, 0, 0, 0), (3, 3, 0, 0...","(0.2, 0.13333333333333, 0.06666666666667, 0.06...","(6, 4, 2, 2, 1)","(15, 6, 4, 2, 2, 1)",30
7,"((2, 2, 2, 2, 2), (5, 0, 0, 0, 0), (3, 3, 0, 0...","(0.2, 0.13333333333333, 0.13333333333333, -0.0...","(6, 4, 4, 0, 1)","(15, 6, 4, 4, 0, 1)",30
8,"((2, 2, 2, 2, 2), (5, 0, 0, 0, 0), (3, 3, 1, 0...","(0.2, 0.13333333333333, 0.0, 0.13333333333333,...","(6, 4, 0, 4, 1)","(15, 6, 4, 0, 4, 1)",30
9,"((2, 2, 2, 2, 2), (5, 0, 0, 0, 0), (3, 3, 1, 0...","(0.2, 0.1, 0.1, 0.1, 0.0)","(2, 1, 1, 1, 0)","(5, 2, 1, 1, 1, 0)",10


## Paarbildung aus den x-basiert erzeugten 5er-WS

In [7]:
single_5_weights_complete = sorted(set(
    tuple(int(x) for x in w)
    for q, w in zip(
        df_single5_x_complete["q_tilde"],
        df_single5_x_complete["weights_5_reduced"]
    )
    if len(w) == 5
    and all(float(x) > 1e-12 for x in q)
    and all(int(x) > 0 for x in w)
))

rows = []
seen = set()
target = np.ones(7, dtype=int)

start_time = time.time()

for i, w1 in enumerate(single_5_weights_complete):
    if i % 20 == 0:
        print(f"Fortschritt: Baustein {i}/{len(single_5_weights_complete)}...")

    for w2 in single_5_weights_complete[i:]:
        full_mat = build_55_overlap3_from_5weights(w1, w2)
        key = canonicalize_rows(full_mat)

        if key in seen:
            continue
        seen.add(key)

        info = analyze_cws_candidate(key)
        if info is None:
            continue

        rows.append({
            "type": "5+5",
            "subtype": "overlap_3",
            "row1": key[0],
            "row2": key[1],
            **info,
        })
end_time = time.time()
duration_1 = end_time - start_time
print(f"Methode 1 (Filter): {duration_1:.4f} Sekunden")

df_55_overlap_3 = pd.DataFrame(rows)
df_55_overlap_3.to_csv("cws_55_overlap_3_from_x_based_singles.csv", index=False)
print("Gespeichert: cws_55_overlap_3_from_x_based_singles.csv")
print("Anzahl Zeilen:", len(df_55_overlap_3))
df_55_overlap_3.head(20)

Fortschritt: Baustein 0/23...
Fortschritt: Baustein 20/23...
Methode 1 (Filter): 87.0631 Sekunden
Gespeichert: cws_55_overlap_3_from_x_based_singles.csv
Anzahl Zeilen: 98


,type,subtype,row1,row2,n_lattice_points,dim,reflexive,pts5d_count
0,5+5,overlap_3,"(2, 1, 1, 0, 0, 1, 1)","(2, 1, 1, 1, 1, 0, 0)",462,5,True,462
1,5+5,overlap_3,"(2, 1, 1, 1, 1, 0, 0)","(2, 2, 1, 0, 0, 1, 1)",445,5,True,445
2,5+5,overlap_3,"(2, 1, 1, 1, 1, 0, 0)","(3, 1, 1, 0, 0, 1, 1)",535,5,True,535
3,5+5,overlap_3,"(2, 1, 1, 1, 1, 0, 0)","(3, 2, 2, 0, 0, 1, 1)",481,5,True,481
4,5+5,overlap_3,"(2, 1, 1, 1, 1, 0, 0)","(4, 1, 1, 0, 0, 2, 1)",399,5,True,399
5,5+5,overlap_3,"(2, 1, 1, 1, 1, 0, 0)","(4, 2, 1, 0, 0, 1, 1)",581,5,False,581
6,5+5,overlap_3,"(2, 1, 1, 1, 1, 0, 0)","(4, 2, 1, 0, 0, 1, 2)",390,5,True,390
7,5+5,overlap_3,"(2, 1, 1, 1, 1, 0, 0)","(4, 2, 1, 0, 0, 2, 1)",390,5,True,390
8,5+5,overlap_3,"(2, 1, 1, 1, 1, 0, 0)","(4, 2, 3, 0, 0, 1, 1)",546,5,True,546
9,5+5,overlap_3,"(2, 1, 1, 1, 1, 0, 0)","(4, 2, 3, 0, 0, 2, 1)",364,5,True,364
